# D.R.O.N.A. — 03 · Feature Engineering: the dual-embedding space

In this project the learned features are **dense sentence embeddings** plus
**BM25 sparse features** - there is no hand-crafted tabular feature matrix.
This notebook builds both, inspects their quality, runs the ablation that
justifies the *dual*-embedding design (research contribution C1), and ingests
everything into ChromaDB for retrieval.

```
01 data cleaning  ->  02 EDA  ->  03 feature engineering  ->  04 model training  ->  05 evaluation
   & preprocessing                   (dual embeddings)          (Colab A100)          & comparison
```

**Runtime:** any GPU accelerates encoding (A100 ≈ seconds); CPU works too.
**Inputs:** artifacts from notebook 01. **Outputs:** `data/processed/embeddings/*.npy`,
populated ChromaDB at `data/chromadb/`, figures in `reports/figures/03_features/`.

## Why two embedding models?

| Collection | Model | Why |
|---|---|---|
| curriculum | [`BAAI/bge-small-en-v1.5`](https://huggingface.co/BAAI/bge-small-en-v1.5) | Top-ranked small general/academic text encoder on MTEB; module descriptors are academic prose |
| career | [`TechWolf/JobBERT-v2`](https://huggingface.co/TechWolf/JobBERT-v2) | Trained specifically on job postings and skill text; understands "React dev, 2yrs exp, Kathmandu" better than a general model |

A single shared model would force one space to compromise. The ablation in §6
measures exactly what is gained.

## 0 · Colab setup

In [ ]:
# ===========================================================================
#  D.R.O.N.A. - Colab setup.  *** RUN THIS CELL FIRST ***
#  Target: Google Colab with an A100 GPU (Runtime > Change runtime type).
#  GPU optional - encoding is ~20x faster on the A100 but CPU works.
# ===========================================================================
import os, sys, subprocess, pathlib

gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
print(gpu or "No GPU detected - see the note at the top of this notebook.")

# EDIT this to your GitHub repo URL. Private repo? use a fine-grained read
# token: https://<TOKEN>@github.com/<user>/D.R.O.N.A.git
# Alternatively upload a zip of the project (Colab Files panel) or attach it
# as a Kaggle dataset named 'drona' - the search loop below finds it.
REPO_URL = "https://github.com/<your-username>/D.R.O.N.A.git"   # <-- EDIT ME

def _is_repo(p):
    return pathlib.Path(p, "drona", "__init__.py").is_file()

search = [".", "..", "../..", "D.R.O.N.A", "/content/D.R.O.N.A",
          "/kaggle/working/D.R.O.N.A", "/kaggle/input/drona/D.R.O.N.A", "/kaggle/input/drona"]
repo = next((p for p in search if _is_repo(p)), None)
if repo is None and "<your-username>" not in REPO_URL:
    dest = "/content/D.R.O.N.A" if pathlib.Path("/content").is_dir() else "D.R.O.N.A"
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, dest], check=True)
    repo = dest
assert repo and _is_repo(repo), (
    "Repo not found. Set REPO_URL to your GitHub URL, OR upload/attach the "
    "project, then re-run. See docs/COLAB_TRAINING_GUIDE.md.")
os.chdir(repo)
print("repo:", os.getcwd())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[eval]"], check=False)

# If Colab cannot reach huggingface.co (downloads stall forever at N%), set
# this True to route all model downloads through the hf-mirror.com mirror.
USE_HF_MIRROR = False
if USE_HF_MIRROR:
    os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
    print("routing HF downloads via hf-mirror.com")

# Robust, resumable HF downloads - plain Colab downloads stall mid-file and
# restart lower on retry. hf_transfer pulls in parallel chunks and resumes.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "hf_transfer"], check=False)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"
# Use an HF token from Colab Secrets (key icon: name it HF_TOKEN) to avoid the
# anonymous rate limits that throttle downloads on Colab's shared IPs.
try:
    from google.colab import userdata as _ud
    _tok = _ud.get("HF_TOKEN")
    if _tok:
        os.environ["HF_TOKEN"] = _tok
        print("HF token loaded from Colab Secrets - authenticated downloads")
except Exception:
    pass
print("setup complete - continue to the next cell")

In [ ]:
# ── Load & VERIFY the real private data (curriculum) — RUN AFTER SETUP ────────
# The real curriculum ships separately in drona_private_data.zip (it is
# gitignored). This cell puts it into the repo and HARD-STOPS on placeholder
# data, so you never train on dummies. It uses a DIRECT in-notebook upload
# (no Google Drive needed): the file picker BLOCKS this cell until you choose the
# zip, so later cells cannot run without the data. Re-running is safe — once the
# data is present it does nothing.
import glob, os, pathlib, subprocess

REPO = next((p for p in (".", "/content/D.R.O.N.A", "D.R.O.N.A", "..")
             if pathlib.Path(p, "drona", "__init__.py").is_file()), ".")

def _real_curriculum_count():
    d = pathlib.Path(REPO, "data/raw/curriculum")
    files = list(d.glob("*.md")) if d.is_dir() else []
    return sum(1 for f in files
               if "DUMMY / PLACEHOLDER" not in f.read_text(encoding="utf-8", errors="ignore")[:200])

def _unzip(z):
    print("unzipping", z, "...")
    subprocess.run(["unzip", "-oq", z, "-d", REPO], check=True)

if _real_curriculum_count() >= 40:
    print(f"real curriculum already present ({_real_curriculum_count()} modules) - OK to proceed")
else:
    # Reuse a bundle already on disk (Files-panel upload, Drive, or a prior run).
    found = (glob.glob("drona_private_data.zip")
             + glob.glob("/content/drona_private_data.zip")
             + glob.glob(f"{REPO}/drona_private_data.zip")
             + glob.glob("../drona_private_data.zip")
             + glob.glob("/content/drive/MyDrive/**/drona_private_data.zip", recursive=True))
    if found:
        _unzip(found[0])
    else:
        try:
            from google.colab import files
        except ImportError:
            files = None
        if files is not None:
            print("\n" + "=" * 64)
            print(">>> Upload drona_private_data.zip now — click 'Choose Files'. <<<")
            print("    It is in your project folder on your PC (~4 MB).")
            print("=" * 64 + "\n")
            uploaded = files.upload()          # BLOCKS until the upload finishes
            zname = next((n for n in uploaded if n.lower().endswith(".zip")), None)
            if zname:
                if not os.path.exists(zname):
                    pathlib.Path(zname).write_bytes(uploaded[zname])
                _unzip(zname)

    n = _real_curriculum_count()
    if n < 40:
        raise SystemExit(
            f"\n*** REAL CURRICULUM NOT LOADED (only {n} real module files) ***\n"
            "Re-run THIS cell and upload drona_private_data.zip when the picker appears.")
    print(f"real curriculum loaded: {n} module files across all programmes - OK to proceed")


In [ ]:
# ── Sentence encoders: upload the OFFLINE bundle directly (no HF download) ─────
# If Colab can't reach Hugging Face, load the encoders from drona_encoder_models.zip
# (built on your PC, ~490MB). This reuses the bundle if it's already cached/on disk;
# otherwise the file picker BELOW BLOCKS this cell until you upload it. Offline mode
# is then set so the SentenceTransformer(...) cell needs ZERO download. To use HF
# instead, just don't upload (leave the picker empty) - it falls back to HF
# (HF_TOKEN secret + hf_transfer / USE_HF_MIRROR from setup).
import os, glob, subprocess, pathlib

_cache = pathlib.Path.home() / ".cache/huggingface"
_need = ["models--BAAI--bge-small-en-v1.5", "models--TechWolf--JobBERT-v2"]

def _encoders_cached():
    return all((_cache / "hub" / m).is_dir() for m in _need)

def _unzip(z):
    _cache.mkdir(parents=True, exist_ok=True)
    print("unzipping", z, "...")
    subprocess.run(["unzip", "-oq", z, "-d", str(_cache)], check=True)

if _encoders_cached():
    print("encoders already in cache - nothing to upload")
else:
    found = (glob.glob("drona_encoder_models.zip")
             + glob.glob("/content/drona_encoder_models.zip")
             + glob.glob("../drona_encoder_models.zip"))
    if found:
        _unzip(found[0])
    else:
        try:
            from google.colab import files
        except ImportError:
            files = None
        if files is not None:
            print("\n" + "=" * 66)
            print(">>> Upload drona_encoder_models.zip (~490MB) - click 'Choose Files'. <<<")
            print("    Big file: if the picker stalls, cancel, drag the zip into the")
            print("    Files panel (folder icon, left) instead, then re-run this cell.")
            print("=" * 66 + "\n")
            up = files.upload()          # BLOCKS until the upload finishes
            zname = next((n for n in up if n.lower().endswith(".zip")), None)
            if zname:
                if not os.path.exists(zname):
                    pathlib.Path(zname).write_bytes(up[zname])
                _unzip(zname)

if _encoders_cached():
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"
    print("encoders ready OFFLINE ->", [p.name for p in (_cache / "hub").glob("models--*")])
else:
    print("no encoder bundle provided - encoders will download from HF instead "
          "(set the HF_TOKEN secret, or USE_HF_MIRROR=True in setup)")


In [ ]:
# Ensure the processed artifacts from notebook 01 exist; regenerate if missing.
import pathlib, subprocess, sys

needed = ["data/processed/onet_career_pathways.parquet",
          "data/processed/onet_career_pathways.json",
          "data/processed/curriculum_modules.json",
          "data/processed/manual_postings.json",
          "data/finetune/sft_train.jsonl",
          "data/demonstrations/demonstrations.jsonl"]
missing = [p for p in needed if not pathlib.Path(p).exists()]
if missing:
    print("missing artifacts:", *missing, sep="\n  ")
    print("\nregenerating with scripts/prepare_training_data.py "
          "(or run notebook 01 for the audited version) ...")
    subprocess.run([sys.executable, "scripts/prepare_training_data.py"], check=True)
print("all pipeline inputs present")

In [ ]:
# --- Shared plotting style: colorblind-safe palette, publication defaults ----
import random, pathlib
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# Categorical palette - CVD-validated ordering. Never reorder or cycle it;
# a 9th series folds into "other".
C = ["#2a78d6", "#1baf7a", "#eda100", "#008300",
     "#4a3aa7", "#e34948", "#e87ba4", "#eb6834"]
INK = {"primary": "#0b0b0b", "secondary": "#52514e", "muted": "#898781",
       "grid": "#e1e0d9", "axis": "#c3c2b7", "surface": "#fcfcfb"}
SEQ = LinearSegmentedColormap.from_list("drona_seq",
    ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"])
DIV = LinearSegmentedColormap.from_list("drona_div",
    ["#104281", "#5598e7", "#f0efec", "#e88a89", "#a52827"])

mpl.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "figure.facecolor": INK["surface"], "axes.facecolor": INK["surface"],
    "axes.edgecolor": INK["axis"], "axes.linewidth": 0.8,
    "axes.grid": True, "grid.color": INK["grid"], "grid.linewidth": 0.6,
    "axes.axisbelow": True, "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlelocation": "left", "axes.titleweight": "bold", "axes.titlesize": 12,
    "axes.labelcolor": INK["secondary"], "axes.labelsize": 10,
    "xtick.color": INK["muted"], "ytick.color": INK["muted"],
    "xtick.labelsize": 9, "ytick.labelsize": 9,
    "text.color": INK["primary"], "font.family": "sans-serif",
    "legend.frameon": False, "legend.fontsize": 9,
    "axes.prop_cycle": mpl.cycler(color=C),
})

FIG_DIR = pathlib.Path("reports/figures/03_features")
FIG_DIR.mkdir(parents=True, exist_ok=True)

def finish(ax, title, subtitle=None, xlabel=None, ylabel=None, grid_axis="y"):
    """Standard title/label treatment. grid_axis: which axis keeps gridlines."""
    ax.set_title(title, pad=22 if subtitle else 8)
    if subtitle:
        ax.text(0, 1.03, subtitle, transform=ax.transAxes,
                fontsize=9, color=INK["secondary"])
    if xlabel: ax.set_xlabel(xlabel)
    if ylabel: ax.set_ylabel(ylabel)
    ax.grid(visible=False, axis="x" if grid_axis == "y" else "y")
    return ax

def save_fig(fig, name):
    p = FIG_DIR / f"{name}.png"
    fig.savefig(p)
    print(f"figure saved -> {p}")

print("plot style ready; figures ->", FIG_DIR)

## 1 · Build the document corpora

Document construction mirrors `drona/data_pipeline/ingest.py` so what we
analyse here is what retrieval actually indexes: title + description + skills
per record.

In [ ]:
import json
from pathlib import Path

pathways = json.loads(Path("data/processed/onet_career_pathways.json").read_text(encoding="utf-8"))
modules  = json.loads(Path("data/processed/curriculum_modules.json").read_text(encoding="utf-8"))
postings = json.loads(Path("data/processed/manual_postings.json").read_text(encoding="utf-8"))

cur_texts, cur_meta = [], []
for m in modules:
    cur_texts.append(f"{m['title']}. {m['description']} "
                     f"Skills developed: {', '.join(m['skills_developed'])}.")
    cur_meta.append({"id": m["module_code"], "kind": "module", "label": m["title"]})

car_texts, car_meta = [], []
for p in pathways:
    car_texts.append(f"{p['title']}. {p['description']} "
                     f"Typical skills: {', '.join(p['typical_skills'])}.")
    car_meta.append({"id": p["pathway_id"], "kind": "pathway", "label": p["title"]})
for j in postings:
    car_texts.append(f"{j['title']} at {j.get('employer') or 'unknown'}. "
                     f"Skills: {', '.join(j['skills_required'])}. {j['description'][:400]}")
    car_meta.append({"id": j["posting_id"], "kind": "posting", "label": j["title"]})

print(f"curriculum docs: {len(cur_texts)} | career docs: {len(car_texts)} "
      f"({sum(1 for m in car_meta if m['kind']=='pathway')} pathways + "
      f"{sum(1 for m in car_meta if m['kind']=='posting')} postings)")

## 2 · Encode with both models

In [ ]:
import time
import torch
from sentence_transformers import SentenceTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("encoding on:", DEVICE)

t0 = time.time()
bge     = SentenceTransformer("BAAI/bge-small-en-v1.5", device=DEVICE)
jobbert = SentenceTransformer("TechWolf/JobBERT-v2",     device=DEVICE)
print(f"models loaded in {time.time()-t0:.1f}s")

t0 = time.time()
E_cur = bge.encode(cur_texts, normalize_embeddings=True, show_progress_bar=False)
E_car = jobbert.encode(car_texts, normalize_embeddings=True, show_progress_bar=False)
E_car_bge = bge.encode(car_texts, normalize_embeddings=True, show_progress_bar=False)  # for the §6 ablation
print(f"encoded {len(cur_texts)+2*len(car_texts)} docs in {time.time()-t0:.1f}s")
print(f"curriculum dim={E_cur.shape[1]} | career dim={E_car.shape[1]} (unit-normalised)")

In [ ]:
EMB = Path("data/processed/embeddings"); EMB.mkdir(parents=True, exist_ok=True)
np.save(EMB / "curriculum_bge.npy", E_cur)
np.save(EMB / "career_jobbert.npy", E_car)
pd.DataFrame(cur_meta).to_parquet(EMB / "curriculum_meta.parquet")
pd.DataFrame(car_meta).to_parquet(EMB / "career_meta.parquet")
print("embedding matrices + metadata saved ->", EMB)

## 3 · Projection: does the space organise sensibly?

PCA (global structure, linear) and t-SNE (local neighbourhoods) of the career
space, coloured by document kind. Healthy sign: postings cluster near related
pathways rather than in a separate blob - that is what lets a student query
bridge both.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

kinds = [m["kind"] for m in car_meta]
kind_color = {"pathway": C[0], "posting": C[2]}

p2 = PCA(n_components=2, random_state=SEED).fit_transform(E_car)
t2 = TSNE(n_components=2, random_state=SEED,
          perplexity=min(30, max(5, len(E_car) // 5)), init="pca").fit_transform(E_car)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6))
for ax, pts, name in [(axes[0], p2, "PCA"), (axes[1], t2, "t-SNE")]:
    for kind in ("pathway", "posting"):
        sel = np.array([k == kind for k in kinds])
        ax.scatter(pts[sel, 0], pts[sel, 1], s=42, color=kind_color[kind],
                   alpha=0.85, edgecolors=INK["surface"], linewidths=1.2, label=kind)
        cx, cy = pts[sel, 0].mean(), pts[sel, 1].mean()
        ax.text(cx, cy, kind, fontsize=10, fontweight="bold", ha="center",
                color=kind_color[kind])
    finish(ax, f"{name} of the career space (JobBERT-v2)", grid_axis="y")
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(visible=False)
axes[0].legend(loc="best")
fig.tight_layout()
save_fig(fig, "career_space_projection")
plt.show()

## 4 · Similarity structure

Cosine-similarity heatmaps inside each collection. Curriculum modules are
ordered by programme year - visible year-blocks mean the embedding tracks the
academic progression.

In [ ]:
mod_df = pd.json_normalize(modules)
order = mod_df.sort_values(["year", "module_code"]).index.to_numpy()
S_cur = E_cur @ E_cur.T

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
im0 = axes[0].imshow(S_cur[np.ix_(order, order)], cmap=SEQ, vmin=0.4, vmax=1.0)
axes[0].set_xticks([]); axes[0].set_yticks(range(len(order)),
    mod_df.loc[order, "module_code"], fontsize=7)
finish(axes[0], "Curriculum self-similarity (bge)", subtitle="modules ordered by year",
       grid_axis="x")
axes[0].grid(visible=False)
fig.colorbar(im0, ax=axes[0], shrink=0.85, label="cosine")

pw_idx = [i for i, m in enumerate(car_meta) if m["kind"] == "pathway"][:25]
S_car = E_car[pw_idx] @ E_car[pw_idx].T
im1 = axes[1].imshow(S_car, cmap=SEQ, vmin=0.4, vmax=1.0)
axes[1].set_xticks([]); axes[1].set_yticks(range(len(pw_idx)),
    [car_meta[i]["label"][:28] for i in pw_idx], fontsize=7)
finish(axes[1], "Career pathway self-similarity (JobBERT)", grid_axis="x")
axes[1].grid(visible=False)
fig.colorbar(im1, ax=axes[1], shrink=0.85, label="cosine")

fig.tight_layout()
save_fig(fig, "similarity_heatmaps")
plt.show()

## 5 · Retrieval sanity check

Five representative student queries against both collections - eyeball that
the nearest neighbours are the documents a human advisor would pull.

In [ ]:
queries = [
    "What modules teach Python programming?",
    "machine learning career in Nepal",
    "web developer jobs Kathmandu",
    "which subjects prepare me for cybersecurity",
    "data analyst salary and required skills",
]
q_cur = bge.encode(queries, normalize_embeddings=True)
q_car = jobbert.encode(queries, normalize_embeddings=True)

rows = []
for qi, q in enumerate(queries):
    top_c = np.argsort(q_cur[qi] @ E_cur.T)[::-1][:3]
    top_k = np.argsort(q_car[qi] @ E_car.T)[::-1][:3]
    rows.append({
        "query": q,
        "top curriculum hits": " | ".join(cur_meta[i]["label"][:34] for i in top_c),
        "top career hits": " | ".join(car_meta[i]["label"][:34] for i in top_k),
    })
pd.set_option("display.max_colwidth", 110)
pd.DataFrame(rows)

## 6 · Ablation: is the job-specialised model worth it?

Both models encode the *same* career corpus; the score is **separation** -
mean intra-kind cosine minus mean inter-kind cosine (pathways vs postings).
Higher = the space distinguishes document roles more cleanly, which is what
Reciprocal Rank Fusion exploits when merging ranked lists.

In [ ]:
def separation(E, kinds):
    E = np.asarray(E)
    kinds = np.asarray(kinds)
    S = E @ E.T
    intra, inter = [], []
    for a in range(len(E)):
        for b in range(a + 1, len(E)):
            (intra if kinds[a] == kinds[b] else inter).append(S[a, b])
    return float(np.mean(intra) - np.mean(inter)), float(np.mean(intra)), float(np.mean(inter))

sep_job, intra_j, inter_j = separation(E_car, kinds)
sep_bge, intra_b, inter_b = separation(E_car_bge, kinds)

fig, ax = plt.subplots(figsize=(6.4, 3.4))
models = ["JobBERT-v2\n(career-specialised)", "bge-small-en-v1.5\n(general)"]
vals = [sep_job, sep_bge]
bars = ax.bar(models, vals, color=[C[0], C[1]], width=0.45)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.002, f"{v:.3f}",
            ha="center", fontsize=10, fontweight="bold", color=INK["primary"])
finish(ax, "Career-space separation by encoder",
       subtitle="mean intra-kind minus inter-kind cosine on the identical corpus - higher is better",
       ylabel="separation")
save_fig(fig, "dual_embedding_ablation")
plt.show()

winner = "JobBERT-v2" if sep_job > sep_bge else "bge-small"
print(f"JobBERT separation={sep_job:.3f} (intra {intra_j:.3f} / inter {inter_j:.3f})")
print(f"bge     separation={sep_bge:.3f} (intra {intra_b:.3f} / inter {inter_b:.3f})")
print(f"-> {winner} organises the career corpus better; the dual design keeps "
      "bge where it wins (academic prose) and JobBERT where it wins (job text).")

## 7 · BM25 sparse features - why hybrid retrieval

Dense embeddings can miss exact terms (module codes, tool names); BM25 never
does but has no semantics. The retriever fuses both with Reciprocal Rank
Fusion. A quick side-by-side on one query shows the complementarity.

In [ ]:
from rank_bm25 import BM25Okapi

tokenised = [t.lower().split() for t in cur_texts]
bm25 = BM25Okapi(tokenised)

q = "python programming module"
scores = bm25.get_scores(q.lower().split())
bm_top = np.argsort(scores)[::-1][:5]
dn_top = np.argsort(bge.encode([q], normalize_embeddings=True)[0] @ E_cur.T)[::-1][:5]

side = pd.DataFrame({
    "BM25 top-5": [cur_meta[i]["label"][:40] for i in bm_top],
    "BM25 score": [round(float(scores[i]), 2) for i in bm_top],
    "dense top-5 (bge)": [cur_meta[i]["label"][:40] for i in dn_top],
})
print(f"query: {q!r}")
side

## 8 · Ingest into ChromaDB

`scripts/ingest_data.py` builds the two persistent collections the retriever
serves from, then `--stats-only` verifies the counts.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "scripts/ingest_data.py"], check=True)
subprocess.run([sys.executable, "scripts/ingest_data.py", "--stats-only"], check=True)

## 9 · (Optional) persist to Google Drive

Colab VMs are ephemeral - copy the ChromaDB + embeddings to Drive so notebook
05 can restore them in a fresh session without re-encoding.

In [ ]:
PERSIST_TO_DRIVE = False   # flip to True on Colab to keep the index across sessions

if PERSIST_TO_DRIVE:
    from google.colab import drive
    import shutil
    drive.mount("/content/drive")
    dest = "/content/drive/MyDrive/drona_artifacts"
    shutil.copytree("data/chromadb", f"{dest}/chromadb", dirs_exist_ok=True)
    shutil.copytree("data/processed/embeddings", f"{dest}/embeddings", dirs_exist_ok=True)
    print("persisted ->", dest)
else:
    print("skipped (set PERSIST_TO_DRIVE = True to copy the index to Drive)")

---
**Next:** [04 · Model Training](04_model_training.ipynb) - fine-tune the
advising LLM and train the gesture policies on the A100.